1.定义工具，使用Pydantic定义args_schema

In [4]:

from typing import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from rich import print as rprint

class weather(BaseModel) :
    city: str = Field(
        default="beijing",
        description="具体城市名称"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )


@tool(args_schema=weather)
def get_weather(city : str):
    """获取天气工具"""
    return "天气晴朗！！！"


rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气工具',
        'parameters': {
            'properties': {
                'city': {'default': 'beijing', 'description': '具体城市名称', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                }
            },
            'type': 'object'
        }
    }
}

1.2 使用json Schema格式

In [6]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 需要自定义下面参数
weather = {
    "type": "object",
    "properties": {
    "location": {"type": "string"},
    "units": {"type": "string"},
    "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}


@tool(args_schema=weather)
def get_weather(city : str):
    """获取天气工具"""
    return "天气晴朗！！！"


rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气工具',
        'parameters': {
            'type': 'object',
            'properties': {
                'location': {'type': 'string'},
                'units': {'type': 'string'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['location', 'units', 'include_forecast']
        }
    }
}

2.0 工具调用

In [10]:

import os

from langchain.tools import tool
from typing import Literal

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages.human import HumanMessage


class weather(BaseModel) :
    city: str = Field(
        default="beijing",
        description="具体城市名称"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )


@tool(args_schema=weather)
def get_weather(city : str, unit : Literal["celsius", "fahrenheit"] = "celsius"):
    """获取天气工具"""
    return "天气晴朗！！！"

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

deepseek_res = ChatDeepSeek(
    model_name=DEEPSEEK_MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE,
)

# 1.加载工具
model = deepseek_res.bind_tools([get_weather])

# 2.组装消息
message = [
    HumanMessage(content="杭州今天天气怎么样?")
]

# 3.调用模型
res = model.invoke(message)

# 4.工具消息存储
message.append(res)

# 5.手动调用工具
calls = res.tool_calls
for call in calls:
    if call["name"] == "get_weather":
        re = get_weather.invoke(call)
        message.append(re)

fin = model.invoke(message)
message.append(fin)

# 打印消息
for msg in message :
    msg.pretty_print()

================================ Human Message =================================

杭州今天天气怎么样?
================================== Ai Message ==================================

我来为您查询杭州今天的天气情况。
Tool Calls:
  get_weather (call_00_fNikbkJjJiUexMo2LS3l1911)
 Call ID: call_00_fNikbkJjJiUexMo2LS3l1911
  Args:
    city: hangzhou
================================= Tool Message =================================
Name: get_weather

天气晴朗！！！
================================== Ai Message ==================================

杭州今天天气晴朗，是个不错的天气！☀️

您可以放心安排外出活动。如果还需要了解气温、湿度等更多详细信息，随时告诉我，我可以帮您进一步查询。


2. 多工具调用

In [14]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
        "苹果发布新款iPhone，股价上涨3%",
        "苹果与欧盟达成反垄断和解协议",
        "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
        "微软Azure云业务季度增长超预期",
        "微软完成对Nuance的收购",
        "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
        "谷歌发布新AI模型，性能提升20%",
        "谷歌与OpenAI合作，开发新的AI助手",
        "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)
message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)
    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break
    # 如果有调用工具，处理工具调用响应

    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)
# print("response", response)
# print(response.content)
for msg in message_list:
    msg.pretty_print()

stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_00_gtpF20NVfE2luLbtMRom2153'
news_result content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_01_dg5YHHclIbz6rflj7a9B1072'
没有工具调用，直接返回答案
content='为您查询到以下信息：\n\n## 📈 苹果公司今日股价\n- **当前股价：185.2美元**\n\n## 📰 最近新闻\n1. **苹果发布新款iPhone，股价上涨3%** — 新品发布带动了股价的积极表现。\n2. **苹果与欧盟达成反垄断和解协议** — 公司解决了与欧盟在反垄断方面的争议。\n3. **苹果将在印度扩大生产规模** — 苹果计划进一步拓展在印度的制造布局。\n\n整体来看，苹果最近在产品和业务布局上动作频繁，股价表现也较为强劲。如需了解某条新闻的更多细节，随时告诉我！' additional_kwargs={'refusal': None, 'reasoning_content': ''} response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 665, 'total_tokens': 798, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 512}, 'prompt_cache_hit_tokens': 512, 'prompt_cache_miss_tokens':

2. 工具是否调用参参数设置（tool_choice)none:不调用任何工具；auto：大模型自主选择；required：大模型必须调用工具

In [30]:

import os

from langchain.tools import tool
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models.base import init_chat_model
from langchain_core.messages.human import HumanMessage
from rich import print as rprint


class weather(BaseModel) :
    city: str = Field(
        default="beijing",
        description="具体城市名称"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )


@tool(args_schema=weather)
def get_weather(city : str, unit : Literal["celsius", "fahrenheit"] = "celsius"):
    """获取天气工具"""
    return "天气晴朗！！！"

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")


model = init_chat_model(
    model="openai:qwen-vl-plus",
    api_key = DASHSCOPE_API_KEY,
    base_url = DASHSCOPE_BASE_URL,
)

# 1.加载工具
models = model.bind_tools([get_weather],tool_choice = "auto")

# 2.组装消息
message = [
    HumanMessage(content="杭州今天天气怎么样?")
]

# 3.调用模型
res = models.invoke(message)

rprint(res)

AIMessage(
    content='抱歉，我无法提供实时的天气信息。不过，你可以通过以下方式获取杭州今天的天气情况：\n\n1. 
**使用天气应用程序**：如“墨迹天气”、“彩云天气”等。\n2. 
**访问天气网站**：如中国天气网（www.weather.com.cn）或中央气象台官网。\n3. 
**语音助手**：如果你有智能音箱或手机，可以问“今天杭州天气怎么样？”\n4. 
**搜索引擎**：在百度、谷歌等搜索引擎中输入“杭州今天天气”即可。\n\n希望这些方法能帮到你！如果还有其他问题，欢迎随时
问我。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 129,
            'prompt_tokens': 13,
            'total_tokens': 142,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': None,
                'rejected_prediction_tokens': None,
                'text_tokens': 129
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'qwen-vl-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-f2c089da-d764-9548-bb2b-029fd0761af2',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019ff505-dd05-7162-a306-ffc2c30204c0-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 13,
        'output_tokens': 129,
        'total_tokens': 142,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)

2.1 强制调用特定工具

In [42]:

import os

from langchain.tools import tool
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models.base import init_chat_model
from langchain_core.messages.human import HumanMessage
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city : str) -> str:
    """获取天气工具

    Args:
        city : 城市
        """
    return "天气晴朗！！！"

@tool(parse_docstring=True)
def get_weather1(city : str):
    """获取天气工具

    Args:
        city : 具体城市名称
    """
    return "天气晴朗！！！"

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")


model = init_chat_model(
    model="openai:qwen-plus",
    api_key = DASHSCOPE_API_KEY,
    base_url = DASHSCOPE_BASE_URL,
)

# 1.加载工具
models = model.bind_tools([get_weather,get_weather1],tool_choice = "get_weather1")

# 2.组装消息
message = [
    HumanMessage(content="杭州今天天气怎么样?")
]

# 3.调用模型
res = models.invoke(message)

rprint(res)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 10,
            'prompt_tokens': 227,
            'total_tokens': 237,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'qwen-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-50e2bf09-3f2f-91eb-8ab0-25b807b5b2d0',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019ff50f-754a-7b40-8ac7-3339f4ad5f87-0',
    tool_calls=[
        {
            'name': 'get_weather1',
            'args': {'city': '杭州'},
            'id': 'call_3c7463afefbb4108bc10e8',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 227,
        'output_tokens': 10,
        'total_tokens': 237,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)